# 03 — Visualize Predictions

Load model predictions, overlay on images, and plot pitch positions.  
Useful for qualitative evaluation and debugging.

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path
from mmpose.apis import init_model, inference_bottomup
from sskit.camera import image_to_ground, normalize

DATA_ROOT = Path('../data/raw/SoccerNet/SpiideoSynLoc')
ANN_FILE = DATA_ROOT / 'annotations' / 'val.json'
ANN_FHD_FILE = DATA_ROOT / 'annotations_fullhd' / 'val.json'

## Load model

In [ ]:
# Change these to your checkpoint
CONFIG = '../vendor/mmpose/configs/body_bev_position/spiideo_soccernet/yoloxpose_m_4xb64-300e_640.py'
CHECKPOINT = '../models/baselines/yoloxpose_m_4xb64-300e_960_epoch_300.pth'

model = init_model(CONFIG, CHECKPOINT, device='cuda:0')
model.eval()
print('Model loaded')

## Run inference on a single image

In [ ]:
with open(ANN_FILE) as f:
    anns_4k = json.load(f)

IMG_IDX = 0  # Change this to explore different images
img_data = anns_4k['images'][IMG_IDX]
img_path = DATA_ROOT / 'val' / img_data['file_name']

preds = inference_bottomup(model, str(img_path))

cam = np.array(img_data['camera_matrix'])
undist = np.array(img_data['undist_poly'])
shape_4k = (3, img_data['height'], img_data['width'])

detections = []
for r in preds:
    inst = r.pred_instances
    for i in range(len(inst)):
        score = float(inst.bbox_scores[i]) if hasattr(inst, 'bbox_scores') else 0.0
        if score < 0.3:
            continue
        kpts = np.array(inst.keypoints[i])
        bbox = inst.bboxes[i].cpu().numpy() if hasattr(inst, 'bboxes') else None
        u_4k, v_4k = float(kpts[1][0]) * 2.0, float(kpts[1][1]) * 2.0
        pt = normalize(torch.tensor([[u_4k, v_4k]], dtype=torch.float64), shape_4k)
        g = image_to_ground(cam, undist, pt)
        x, y = g[0][0].item(), g[0][1].item()
        if abs(x) > 60 or abs(y) > 45:
            continue
        detections.append({'bbox': bbox, 'keypoints': kpts, 'score': score, 'world': (x, y)})

print(f'{len(detections)} detections (score >= 0.3)')

## Visualize: image with bboxes + pitch positions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Left: image with bounding boxes
img = Image.open(img_path)
axes[0].imshow(img)
for det in detections:
    if det['bbox'] is not None:
        x1, y1, x2, y2 = det['bbox']
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=1, edgecolor='lime', facecolor='none')
        axes[0].add_patch(rect)
    kpt = det['keypoints'][1]  # pelvis_ground
    axes[0].plot(kpt[0], kpt[1], 'r.', markersize=4)
axes[0].set_title(f'{img_data["file_name"]} — {len(detections)} detections')
axes[0].axis('off')

# Right: pitch positions
# Draw pitch outline
pitch = plt.Rectangle((-52.5, -34), 105, 68, fill=False, edgecolor='white', linewidth=2)
center = plt.Circle((0, 0), 9.15, fill=False, edgecolor='white', linewidth=1)
axes[1].add_patch(pitch)
axes[1].add_patch(center)
axes[1].axvline(0, color='white', linewidth=1, alpha=0.5)
axes[1].set_facecolor('#2d8a4e')

# Plot GT positions (if available)
gt_anns = [a for a in anns_4k['annotations'] if a['image_id'] == img_data['id']]
for a in gt_anns:
    if 'position_on_pitch' in a:
        pos = a['position_on_pitch']
        axes[1].plot(pos[0], pos[1], 'wo', markersize=6, alpha=0.5)

# Plot predicted positions
for det in detections:
    x, y = det['world']
    axes[1].plot(x, y, 'r^', markersize=8)

axes[1].set_xlim(-60, 60)
axes[1].set_ylim(-40, 40)
axes[1].set_aspect('equal')
axes[1].set_title('Pitch positions (white=GT, red=predicted)')
axes[1].set_xlabel('X (m)')
axes[1].set_ylabel('Y (m)')

plt.tight_layout()
plt.show()

## Batch: visualize multiple images

In [ ]:
# Run on first N images and show pitch positions
N = 10
all_pred_pos = []
all_gt_pos = []

for idx in range(min(N, len(anns_4k['images']))):
    img_data = anns_4k['images'][idx]
    img_path = DATA_ROOT / 'val' / img_data['file_name']
    preds = inference_bottomup(model, str(img_path))
    cam = np.array(img_data['camera_matrix'])
    undist = np.array(img_data['undist_poly'])
    shape_4k = (3, img_data['height'], img_data['width'])
    
    for r in preds:
        inst = r.pred_instances
        for i in range(len(inst)):
            score = float(inst.bbox_scores[i]) if hasattr(inst, 'bbox_scores') else 0.0
            if score < 0.3:
                continue
            kpts = np.array(inst.keypoints[i])
            u_4k, v_4k = float(kpts[1][0]) * 2.0, float(kpts[1][1]) * 2.0
            pt = normalize(torch.tensor([[u_4k, v_4k]], dtype=torch.float64), shape_4k)
            g = image_to_ground(cam, undist, pt)
            x, y = g[0][0].item(), g[0][1].item()
            if abs(x) < 60 and abs(y) < 45:
                all_pred_pos.append([x, y])
    
    for a in [a for a in anns_4k['annotations'] if a['image_id'] == img_data['id']]:
        if 'position_on_pitch' in a:
            all_gt_pos.append(a['position_on_pitch'][:2])

print(f'{len(all_pred_pos)} predicted, {len(all_gt_pos)} GT positions from {N} images')